# Step 3. Write, and read back

*With ngff-zarr.*

**Goal:** write both images and the transformation between them in one OME-Zarr 0.6 file, and check what a reader finds in it. No registration tool is needed from here on.

In [1]:
import json
import shutil
import zipfile
from pathlib import Path

import ngff_zarr as nz
import numpy as np
from ngff_zarr import CoordinateSystemIdentifier, NgffScene

FIXED, MOVING = "../output/R0.ome.zarr", "../output/R1.ome.zarr"
TRANSFORM = "../output/registration/P000/Transform.ome.zarr"
STORE, SCENE = "../output/pair.ome.zarr", "../output/pair.ozx"

shutil.rmtree(STORE, ignore_errors=True)  # so that this cell can run twice

# Step 2 wrote the transformation as RFC-5 metadata: it is read as the transformation it is.
registration = nz.from_ome_zarr(TRANSFORM, kind="transformation")

# The scene names what it connects: from the coordinates of the fixed image to those of the moving one.
registration.input = CoordinateSystemIdentifier(path="fixed", name="intrinsic")
registration.output = CoordinateSystemIdentifier(path="moving", name="intrinsic")
images = {"fixed": nz.from_ome_zarr(FIXED), "moving": nz.from_ome_zarr(MOVING)}
NgffScene(images=images, coordinateTransformations=[registration]).to_ome_zarr(STORE, overwrite=False)

# One file: the same store, zipped (RFC-9).
nz.write_store_to_zip(STORE, SCENE, version="0.6")
print(f"{SCENE}: {Path(SCENE).stat().st_size / 1e6:.0f} MB")

../output/pair.ozx: 42 MB


`pair.ozx` is one file:

```text
pair.ozx
├── zarr.json       the scene: the transformation, and what it connects
├── fixed/          R0, with its pyramid
└── moving/         R1, with its pyramid
```

- **The transformation is twelve numbers.** Step 2 wrote it as an RFC-5 `affine`, a 3 × 4 matrix over (z, y, x) in micrometers, and the scene carries it in its own metadata. A `displacements` field, from a deformable preset, is an OME-Zarr image of vectors: it is copied below the scene and pointed at by its path, as [write.py](write.py) does for either form.
- **The scene copies the images.** RFC-5 names them by a path inside the scene, so `pair.ozx` holds R0 and R1 with their pyramids: 42 MB, all of it the two images.
- **The scene says what it connects:** from the coordinate system of `fixed` to the one of `moving`.

## Read it back

This is what a reader finds: the file's metadata, and the transformation in full.

In [2]:
scene = NgffScene.from_ome_zarr(SCENE)
for name, multiscales in scene.images.items():
    size = " x ".join(str(extent) for extent in multiscales.images[0].data.shape)
    print(f"{name}: {size} voxels, {len(multiscales.images)} levels")

transformation = scene.coordinateTransformations[0]
print(f"transformation: {transformation.type}, from {transformation.input.path} to {transformation.output.path}")
print(np.array2string(np.asarray(transformation.affine), precision=3, suppress_small=True))

fixed: 50 x 680 x 535 voxels, 3 levels
moving: 50 x 680 x 535 voxels, 3 levels
transformation: affine, from fixed to moving
[[  0.999  -0.013   0.034   0.344]
 [  0.013   1.      0.001 -10.088]
 [ -0.035  -0.001   0.999  37.001]]


The matrix maps a point of R0 to the same point of R1, in micrometers and in the order z, y, x: a rotation of a few degrees, and in the last column the translation, 37 µm along x.

A reader of OME-Zarr 0.6 finds the same thing in `zarr.json`, at the root of the file:

In [3]:
with zipfile.ZipFile(SCENE) as archive:
    root = json.loads(archive.read("zarr.json"))
print(json.dumps(root["attributes"]["ome"]["scene"]["coordinateTransformations"], indent=2))

[
  {
    "input": {
      "path": "fixed",
      "name": "intrinsic"
    },
    "output": {
      "path": "moving",
      "name": "intrinsic"
    },
    "type": "affine",
    "affine": [
      [
        0.999316790401152,
        -0.013299107171981128,
        0.03448312878449811,
        0.3436142498125642
      ],
      [
        0.01325527158319062,
        0.9999110206388373,
        0.0014995266767758864,
        -10.088313475502503
      ],
      [
        -0.0345000028637088,
        -0.0010414189486809278,
        0.9994041551089197,
        37.00061501043516
      ]
    ]
  }
]


## The size of the transformation

| Preset | What step 2 writes | `Transform.ome.zarr` | `pair.ozx` |
|---|---|---|---|
| `Generic_Rigid` | an `affine`: twelve numbers | 4 kB | 42 MB, the two images |
| `ConvexAdam_Composite` | a `displacements` field: float32, on the fixed grid | 170 MB | 217 MB |

The ITK form the preset writes without `Transform:omezarr` tells the same story: an `AffineTransform` of a few kilobytes, or a 437 MB field.

**Next: [Step 4. Apply](4-apply.ipynb)**